# 03 — SQL Analysis
## Anime Streaming Platform Analytics

The star schema now lives in **SQLite** (`database/anime_streaming.db`), built
by `scripts/load_database.py` with typed columns, primary keys, foreign-key
references and indexes on every fact key.

This notebook runs **45 interview-level queries** kept as the single source of
truth in `sql/*.sql` — each one annotated with the *business question* it
answers and the *SQL technique* it demonstrates:

| File | Focus | Queries |
|---|---|---|
| `01_foundations.sql` | GROUP BY, HAVING, CASE, percentages | Q01–Q11 |
| `02_joins_subqueries_ctes.sql` | multi-table JOINs, subqueries, `NOT EXISTS`, CTEs | Q12–Q22 |
| `03_window_functions.sql` | ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD, NTILE, FIRST_VALUE, running totals, rolling averages | Q23–Q36 |
| `04_business_kpis.sql` | executive KPIs: churn hazard, ARPU, MAU/DAU, cohorts, channel quality | Q37–Q45 |

Each cell prints the annotated SQL and shows its result, so the notebook reads
as a worked answer key.

In [1]:
import importlib.util
import re
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Rebuild the database from the processed CSVs on every run (idempotent, ~3 s)
spec = importlib.util.spec_from_file_location("load_database", ROOT / "scripts" / "load_database.py")
loader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(loader)
conn = sqlite3.connect(loader.build())

  dim_user                 7,993 rows
  dim_content                 64 rows


  dim_date                 1,250 rows
  fact_subscriptions       7,993 rows


  fact_watch_events       41,046 rows

Integrity checks:
  [PASS] events with unknown user: 0
  [PASS] events with unknown content: 0
  [PASS] events with unknown date: 0
  [PASS] subscriptions without user: 0

Database ready: F:\Projects\Vibe coding\AI News summarizer\database\anime_streaming.db


In [2]:
QUERIES: dict[int, tuple[str, list[str], str]] = {}

for path in sorted((ROOT / "sql").glob("*.sql")):
    blocks = re.split(r"(?m)^-- Q(\d+) \u00b7 ", path.read_text(encoding="utf-8"))[1:]
    for qid, body in zip(blocks[::2], blocks[1::2]):
        lines = body.splitlines()
        title, notes, i = lines[0].strip(), [], 1
        while i < len(lines) and lines[i].startswith("--"):
            notes.append(lines[i].lstrip("- ").rstrip())
            i += 1
        QUERIES[int(qid)] = (title, notes, "\n".join(lines[i:]).strip())


def run(qid: int) -> pd.DataFrame:
    """Print a query's annotation + SQL, execute it, return the result."""
    title, notes, sql = QUERIES[qid]
    print(f"Q{qid:02d} - {title}")
    for line in notes:
        print(f"  {line}")
    print("\n" + sql + "\n")
    return pd.read_sql_query(sql, conn)


print(f"Parsed {len(QUERIES)} queries from {len(list((ROOT / 'sql').glob('*.sql')))} files")

Parsed 45 queries from 4 files


---
## Part 1 · Foundations — GROUP BY, HAVING, CASE

The bread-and-butter tier: aggregate, filter aggregates, bucket with CASE, and
express results as shares of a total. Most screening interviews live here.

In [3]:
run(1)

Q01 - Table row counts
  Business: sanity-check the load before trusting any query on top of it.
  Technique: UNION ALL to stack scalar aggregates into one result.

SELECT 'dim_user' AS table_name, COUNT(*) AS rows_ FROM dim_user
UNION ALL SELECT 'dim_content', COUNT(*) FROM dim_content
UNION ALL SELECT 'dim_date', COUNT(*) FROM dim_date
UNION ALL SELECT 'fact_subscriptions', COUNT(*) FROM fact_subscriptions
UNION ALL SELECT 'fact_watch_events', COUNT(*) FROM fact_watch_events;



,table_name,rows_
0,dim_user,7993
1,dim_content,64
2,dim_date,1250
3,fact_subscriptions,7993
4,fact_watch_events,41046


In [4]:
run(2)

Q02 - Users, active users and churn rate by plan
  Business: which tiers hold their customers? The one-table health check.
  Technique: conditional aggregation with CASE inside SUM.

SELECT Subscription_Plan,
       COUNT(*)                                                        AS users,
       SUM(CASE WHEN Subscription_Status = 'Active' THEN 1 ELSE 0 END) AS active,
       ROUND(100.0 * SUM(CASE WHEN Subscription_Status = 'Cancelled'
                              THEN 1 ELSE 0 END) / COUNT(*), 1)        AS churn_pct
FROM dim_user
GROUP BY Subscription_Plan
ORDER BY users DESC;



,Subscription_Plan,users,active,churn_pct
0,Free,3195,1102,65.5
1,Basic,2018,822,59.3
2,Premium,1992,1107,44.4
3,Family,788,493,37.4


In [5]:
run(3)

Q03 - Top 10 anime by watch hours
  Business: the renewal shortlist — which licenses earn their fee.
  Technique: fact-to-dimension JOIN, GROUP BY, ORDER BY aggregate, LIMIT.

SELECT c.Anime_Title,
       c.Genre,
       ROUND(SUM(f.Watch_Time_Minutes) / 60.0, 0) AS watch_hours,
       COUNT(*)                                   AS views
FROM fact_watch_events f
JOIN dim_content c USING (Content_ID)
GROUP BY c.Anime_Title, c.Genre
ORDER BY SUM(f.Watch_Time_Minutes) DESC
LIMIT 10;



,Anime_Title,Genre,watch_hours,views
0,One Piece,Shonen,1216.0,5100
1,Attack on Titan,Shonen,737.0,3046
2,Demon Slayer: Kimetsu no Yaiba,Shonen,532.0,2241
3,Jujutsu Kaisen,Shonen,439.0,1857
4,My Hero Academia,Shonen,355.0,1534
5,Naruto Shippuden,Shonen,324.0,1398
6,Solo Leveling,Fantasy,281.0,1174
7,Chainsaw Man,Shonen,247.0,1057
8,Frieren: Beyond Journey's End,Fantasy,233.0,956
9,Spy x Family,Slice of Life,226.0,967


In [6]:
run(4)

Q04 - Most-watched studios
  Business: which studios deserve output deals rather than per-title licensing.
  Technique: JOIN + COUNT(DISTINCT) to measure audience, not just volume.

SELECT c.Studio,
       ROUND(SUM(f.Watch_Time_Minutes) / 60.0, 0) AS watch_hours,
       COUNT(DISTINCT f.User_ID)                  AS unique_viewers,
       COUNT(*)                                   AS views
FROM fact_watch_events f
JOIN dim_content c USING (Content_ID)
GROUP BY c.Studio
ORDER BY watch_hours DESC
LIMIT 10;



,Studio,watch_hours,unique_viewers,views
0,MAPPA,1651.0,3171,6921
1,Toei Animation,1216.0,2631,5100
2,Madhouse,1075.0,2498,4526
3,A-1 Pictures,745.0,1732,3167
4,Studio Pierrot,689.0,1917,2948
5,Bones,664.0,1795,2834
6,Ufotable,532.0,1514,2241
7,CloverWorks,340.0,880,1467
8,Wit Studio,298.0,787,1271
9,Production I.G,295.0,639,1252


In [7]:
run(5)

Q05 - Highest-revenue countries
  Business: where the money lives — prioritise localisation and payments.
  Technique: JOIN subscriptions to user geography; revenue is user-grain.

SELECT u.Country,
       COUNT(*)                                AS users,
       ROUND(SUM(s.Revenue), 0)                AS lifetime_revenue,
       ROUND(SUM(s.Revenue) / COUNT(*), 2)     AS revenue_per_user
FROM fact_subscriptions s
JOIN dim_user u USING (User_ID)
GROUP BY u.Country
ORDER BY lifetime_revenue DESC
LIMIT 10;



,Country,users,lifetime_revenue,revenue_per_user
0,USA,1718,79491.0,46.27
1,Japan,1035,47567.0,45.96
2,India,961,44943.0,46.77
3,Brazil,709,35366.0,49.88
4,Mexico,518,25435.0,49.10
5,UK,474,22503.0,47.48
6,Philippines,457,21650.0,47.37
7,France,399,19456.0,48.76
8,Indonesia,388,19130.0,49.30
9,Canada,421,18338.0,43.56


In [8]:
run(6)

Q06 - Device share of viewing
  Business: which platforms the apps team should prioritise.
  Technique: percentage of total via a scalar subquery.

SELECT Device,
       COUNT(*)                                                            AS events,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM fact_watch_events), 1) AS share_pct
FROM fact_watch_events
GROUP BY Device
ORDER BY events DESC;



,Device,events,share_pct
0,Mobile,20853,50.8
1,Smart TV,8243,20.1
2,Desktop,7144,17.4
3,Tablet,4777,11.6
4,Unknown,29,0.1


In [9]:
run(7)

Q07 - Viewing quality by plan
  Business: do paying users actually watch more and finish more?
  Technique: CASE in ORDER BY to sort a category logically, not alphabetically.

SELECT u.Subscription_Plan,
       COUNT(*)                                  AS events,
       ROUND(AVG(f.Watch_Time_Minutes), 1)       AS avg_watch_min,
       ROUND(AVG(f.Completion_Percentage), 1)    AS avg_completion_pct,
       ROUND(AVG(f.Buffering_Time), 1)           AS avg_buffering_s
FROM fact_watch_events f
JOIN dim_user u USING (User_ID)
GROUP BY u.Subscription_Plan
ORDER BY CASE u.Subscription_Plan
             WHEN 'Free' THEN 1 WHEN 'Basic' THEN 2
             WHEN 'Premium' THEN 3 ELSE 4 END;



,Subscription_Plan,events,avg_watch_min,avg_completion_pct,avg_buffering_s
0,Free,7011,12.4,54.0,8.5
1,Basic,8346,13.7,59.8,8.7
2,Premium,16623,14.7,63.7,8.9
3,Family,9066,14.9,64.9,9.3


In [10]:
run(8)

Q08 - Genres that both scale and hold attention
  Business: candidates for more licensing spend — volume AND completion.
  Technique: HAVING filters on aggregates after GROUP BY (WHERE cannot).

SELECT c.Genre,
       COUNT(*)                               AS views,
       ROUND(AVG(f.Completion_Percentage), 1) AS avg_completion_pct
FROM fact_watch_events f
JOIN dim_content c USING (Content_ID)
GROUP BY c.Genre
HAVING COUNT(*) >= 1500 AND AVG(f.Completion_Percentage) >= 60
ORDER BY views DESC;



,Genre,views,avg_completion_pct
0,Shonen,20555,61.9
1,Isekai,3690,60.2
2,Fantasy,3038,62.2
3,Seinen,2975,61.8
4,Slice of Life,2325,60.7
5,Romance,1644,60.8


In [11]:
run(9)

Q09 - Engagement tiers and their churn
  Business: quantify how strongly engagement protects retention.
  Technique: CASE bucketing of a continuous score into named tiers.

SELECT CASE WHEN Engagement_Score >= 75 THEN 'A · 75-100'
            WHEN Engagement_Score >= 50 THEN 'B · 50-74'
            WHEN Engagement_Score >= 25 THEN 'C · 25-49'
            ELSE 'D · 0-24' END               AS engagement_tier,
       COUNT(*)                               AS users,
       ROUND(100.0 * SUM(CASE WHEN Subscription_Status = 'Cancelled'
                              THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_pct
FROM dim_user
GROUP BY engagement_tier
ORDER BY engagement_tier;



,engagement_tier,users,churn_pct
0,A · 75-100,924,36.0
1,B · 50-74,2938,56.4
2,C · 25-49,3369,59.5
3,D · 0-24,762,62.2


In [12]:
run(10)

Q10 - Why users cancel
  Business: each reason routes to a different owner (pricing, content, infra).
  Technique: WHERE against NULL + share-of-total via subquery.

SELECT Cancellation_Reason,
       COUNT(*) AS cancelled_users,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM fact_subscriptions
                                 WHERE Cancellation_Reason IS NOT NULL), 1) AS share_pct
FROM fact_subscriptions
WHERE Cancellation_Reason IS NOT NULL
GROUP BY Cancellation_Reason
ORDER BY cancelled_users DESC;



,Cancellation_Reason,cancelled_users,share_pct
0,Lost Interest,1257,28.1
1,Technical Issues,1229,27.5
2,Not Enough Content,752,16.8
3,Switched to Competitor,649,14.5
4,Too Expensive,422,9.4
5,Payment Failure,160,3.6


In [13]:
run(11)

Q11 - Audio-language preference by watch time
  Business: sub vs dub investment — where dubbing budgets pay off.
  Technique: aggregate on a joined dimension attribute with share of total.

SELECT u.Language,
       ROUND(SUM(f.Watch_Time_Minutes) / 60.0, 0) AS watch_hours,
       ROUND(100.0 * SUM(f.Watch_Time_Minutes)
             / (SELECT SUM(Watch_Time_Minutes) FROM fact_watch_events), 1) AS share_pct
FROM fact_watch_events f
JOIN dim_user u USING (User_ID)
GROUP BY u.Language
ORDER BY watch_hours DESC;



,Language,watch_hours,share_pct
0,Japanese,5021.0,51.9
1,English,2789.0,28.8
2,Portuguese,549.0,5.7
3,Spanish,403.0,4.2
4,Hindi,302.0,3.1
5,German,259.0,2.7
6,French,238.0,2.5
7,Korean,120.0,1.2


**Part 1 takeaways.** Free/Basic churn hardest while Family holds best (Q02);
the top-10 titles are Shonen-heavy (Q03) and paid plans watch longer *and*
finish more with less buffering pain (Q07). Cancellation reasons split across
price, content and technical owners (Q10) — three different roadmaps hiding in
one column.

---
## Part 2 · Joins, subqueries & CTEs

Multi-table reasoning: revenue must always be pulled from the user-grain
subscriptions fact (never summed at event grain), anti-joins find who *didn't*
do something, and CTEs keep multi-step logic readable.

In [14]:
run(12)

Q12 - Which genre's fans are worth the most
  Business: revenue attribution by taste — the licensing budget compass.
  Technique: CTE assigns each user a dominant genre (ROW_NUMBER over an
  aggregate), then joins to user-grain revenue. Never SUM revenue
  at event grain — it double-counts.

WITH user_genre AS (
    SELECT f.User_ID,
           c.Genre,
           ROW_NUMBER() OVER (PARTITION BY f.User_ID
                              ORDER BY SUM(f.Watch_Time_Minutes) DESC) AS rn
    FROM fact_watch_events f
    JOIN dim_content c USING (Content_ID)
    GROUP BY f.User_ID, c.Genre
)
SELECT ug.Genre                              AS dominant_genre,
       COUNT(*)                              AS fans,
       ROUND(SUM(s.Revenue), 0)              AS lifetime_revenue,
       ROUND(SUM(s.Revenue) / COUNT(*), 2)   AS revenue_per_fan
FROM user_genre ug
JOIN fact_subscriptions s USING (User_ID)
WHERE ug.rn = 1
GROUP BY ug.Genre
ORDER BY lifetime_revenue DESC;



,dominant_genre,fans,lifetime_revenue,revenue_per_fan
0,Shonen,4123,191710.0,46.50
1,Isekai,672,33408.0,49.71
2,Fantasy,588,27235.0,46.32
3,Seinen,553,25589.0,46.27
4,Slice of Life,467,21380.0,45.78
5,Romance,302,14519.0,48.08
6,Sports,240,13205.0,55.02
7,Thriller,287,11532.0,40.18
8,Mecha,215,11129.0,51.76
9,Shojo,192,9455.0,49.24


In [15]:
run(13)

Q13 - Most loyal customers
  Business: the profile of a perfect customer — who to study and to protect.
  Technique: multi-table JOIN with composite ordering.

SELECT u.User_ID, u.Country, u.Subscription_Plan,
       s.Membership_Tenure                    AS tenure_months,
       u.Engagement_Score,
       ROUND(s.Revenue, 0)                    AS lifetime_revenue
FROM dim_user u
JOIN fact_subscriptions s USING (User_ID)
WHERE u.Subscription_Status = 'Active'
ORDER BY s.Membership_Tenure DESC, u.Engagement_Score DESC
LIMIT 15;



,User_ID,Country,Subscription_Plan,tenure_months,Engagement_Score,lifetime_revenue
0,U02421,Brazil,Basic,41,48.3,205.0
1,U07322,Japan,Family,40,78.3,600.0
2,U01785,India,Free,40,47.2,35.0
3,U01160,India,Basic,40,46.6,200.0
4,U07536,Mexico,Free,40,22.4,22.0
5,U02708,South Korea,Family,39,86.8,585.0
6,U04154,UK,Family,39,85.3,585.0
7,U05793,UK,Premium,39,82.4,390.0
8,U01103,Japan,Premium,39,82.1,390.0
9,U06350,USA,Basic,39,65.7,195.0


In [16]:
run(14)

Q14 - Power watchers: users at 2x the average total watch time
  Business: the heavy-usage cohort that stresses infrastructure and loves you.
  Technique: CTE of per-user totals filtered by a scalar subquery on itself.

WITH user_totals AS (
    SELECT User_ID, SUM(Watch_Time_Minutes) AS total_min
    FROM fact_watch_events
    GROUP BY User_ID
)
SELECT COUNT(*)                                          AS power_watchers,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM user_totals), 1) AS pct_of_users,
       ROUND(MIN(total_min) / 60.0, 1)                   AS min_hours,
       ROUND(MAX(total_min) / 60.0, 1)                   AS max_hours
FROM user_totals
WHERE total_min > 2 * (SELECT AVG(total_min) FROM user_totals);



,power_watchers,pct_of_users,min_hours,max_hours
0,1120,14.0,2.4,19.0


In [17]:
run(15)

Q15 - Most completed anime
  Business: completion is the truest quality signal — no self-selection like ratings.
  Technique: HAVING enforces a minimum sample before ranking an average.

SELECT c.Anime_Title, c.Genre,
       COUNT(*)                               AS views,
       ROUND(AVG(f.Completion_Percentage), 1) AS avg_completion_pct
FROM fact_watch_events f
JOIN dim_content c USING (Content_ID)
GROUP BY c.Anime_Title, c.Genre
HAVING COUNT(*) >= 150
ORDER BY avg_completion_pct DESC
LIMIT 10;



,Anime_Title,Genre,views,avg_completion_pct
0,Fullmetal Alchemist: Brotherhood,Shonen,695,63.5
1,Frieren: Beyond Journey's End,Fantasy,956,63.2
2,Attack on Titan,Shonen,3046,63.1
3,Monster,Seinen,471,63.0
4,Bleach: Thousand-Year Blood War,Shonen,847,62.9
5,Haikyu!!,Sports,475,62.9
6,Neon Genesis Evangelion,Mecha,287,62.7
7,Death Note,Thriller,655,62.6
8,Your Lie in April,Romance,383,62.6
9,Oshi no Ko,Seinen,525,62.5


In [18]:
run(16)

Q16 - Highest-rated anime (minimum 100 ratings)
  Business: brand-building titles for marketing to lead with.
  Technique: aggregate over a NULL-heavy column — AVG ignores NULLs; the
  HAVING guard stops tiny samples from topping the chart.

SELECT c.Anime_Title, c.Studio,
       COUNT(f.User_Rating)              AS ratings,
       ROUND(AVG(f.User_Rating), 2)      AS avg_rating
FROM fact_watch_events f
JOIN dim_content c USING (Content_ID)
WHERE f.User_Rating IS NOT NULL
GROUP BY c.Anime_Title, c.Studio
HAVING COUNT(f.User_Rating) >= 100
ORDER BY avg_rating DESC
LIMIT 10;



,Anime_Title,Studio,ratings,avg_rating
0,Fullmetal Alchemist: Brotherhood,Bones,342,8.39
1,Frieren: Beyond Journey's End,Madhouse,437,8.38
2,Attack on Titan,MAPPA,1389,8.31
3,Steins;Gate,White Fox,129,8.26
4,Hunter x Hunter,Madhouse,340,8.22
5,Monster,Madhouse,231,8.20
6,Bleach: Thousand-Year Blood War,Studio Pierrot,380,8.18
7,Death Note,Madhouse,304,8.17
8,Gintama,Sunrise,120,8.12
9,One Piece,Toei Animation,2346,8.11


In [19]:
run(17)

Q17 - Premium revenue by country
  Business: where the highest-value tier concentrates — pricing-test markets.
  Technique: filtered JOIN with per-group ARPU.

SELECT u.Country,
       COUNT(*)                            AS premium_users,
       ROUND(SUM(s.Revenue), 0)            AS lifetime_revenue,
       ROUND(AVG(s.Revenue_Per_Month), 2)  AS avg_monthly_arpu
FROM fact_subscriptions s
JOIN dim_user u USING (User_ID)
WHERE s.Subscription_Plan = 'Premium'
GROUP BY u.Country
ORDER BY lifetime_revenue DESC
LIMIT 10;



,Country,premium_users,lifetime_revenue,avg_monthly_arpu
0,USA,430,38551.0,9.99
1,Japan,265,22827.0,9.99
2,India,234,20559.0,9.99
3,Brazil,174,15325.0,9.99
4,Mexico,138,12777.0,9.99
5,Philippines,130,11678.0,9.99
6,Germany,103,9491.0,9.99
7,France,99,8611.0,9.99
8,Canada,107,8571.0,9.99
9,UK,96,8142.0,9.99


In [20]:
run(18)

Q18 - Broadest-reach titles
  Business: watch hours can be a few superfans; reach = true catalogue anchors.
  Technique: COUNT(DISTINCT) against a scalar-subquery denominator.

SELECT c.Anime_Title,
       COUNT(DISTINCT f.User_ID)                                       AS unique_viewers,
       ROUND(100.0 * COUNT(DISTINCT f.User_ID)
             / (SELECT COUNT(*) FROM dim_user), 1)                     AS reach_pct
FROM fact_watch_events f
JOIN dim_content c USING (Content_ID)
GROUP BY c.Anime_Title
ORDER BY unique_viewers DESC
LIMIT 10;



,Anime_Title,unique_viewers,reach_pct
0,One Piece,2631,32.9
1,Attack on Titan,1845,23.1
2,Demon Slayer: Kimetsu no Yaiba,1514,18.9
3,Jujutsu Kaisen,1350,16.9
4,My Hero Academia,1145,14.3
5,Naruto Shippuden,1088,13.6
6,Chainsaw Man,865,10.8
7,Bleach: Thousand-Year Blood War,714,8.9
8,One Punch Man,711,8.9
9,Solo Leveling,696,8.7


In [21]:
run(19)

Q19 - Active but silent: users with zero interactions
  Business: engagement risk pool — active on paper, disengaged in behaviour.
  Technique: NOT EXISTS anti-join ("Like" is quoted — LIKE is an SQL keyword).

SELECT COUNT(*) AS silent_active_users,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dim_user
                                 WHERE Subscription_Status = 'Active'), 1) AS pct_of_active
FROM dim_user u
WHERE u.Subscription_Status = 'Active'
  AND NOT EXISTS (
        SELECT 1
        FROM fact_watch_events f
        WHERE f.User_ID = u.User_ID
          AND (f."Like" = 1 OR f.Share = 1 OR f.Watchlist = 1 OR f.Download = 1)
      );



,silent_active_users,pct_of_active
0,974,27.6


In [22]:
run(20)

Q20 - Buffering by connection type vs the platform average
  Business: sizes the infra problem per segment instead of one blended number.
  Technique: CROSS JOIN a one-row CTE to compare each group to the whole.

WITH overall AS (
    SELECT AVG(Buffering_Time) AS avg_buffer FROM fact_watch_events
)
SELECT f.Internet_Type,
       COUNT(*)                                   AS events,
       ROUND(AVG(f.Buffering_Time), 1)            AS avg_buffering_s,
       ROUND(AVG(f.Buffering_Time) - o.avg_buffer, 1) AS vs_platform_avg
FROM fact_watch_events f
CROSS JOIN overall o
GROUP BY f.Internet_Type, o.avg_buffer
ORDER BY avg_buffering_s DESC;



,Internet_Type,events,avg_buffering_s,vs_platform_avg
0,Satellite,2066,23.3,14.4
1,Mobile Data,13319,14.7,5.8
2,Broadband,12652,6.3,-2.6
3,Fiber,13009,3.2,-5.7


In [23]:
run(21)

Q21 - Completion profile by device
  Business: the finish-vs-abandon mix per screen — informs autoplay/resume UX.
  Technique: CASE pivot — long categories turned into wide percentage columns.

SELECT Device,
       COUNT(*) AS events,
       ROUND(100.0 * AVG(CASE WHEN Completion_Bucket = 'Completed (90-100%)'
                              THEN 1.0 ELSE 0 END), 1) AS completed_pct,
       ROUND(100.0 * AVG(CASE WHEN Completion_Bucket = 'Abandoned (<25%)'
                              THEN 1.0 ELSE 0 END), 1) AS abandoned_pct
FROM fact_watch_events
GROUP BY Device
ORDER BY completed_pct DESC;



,Device,events,completed_pct,abandoned_pct
0,Smart TV,8243,11.0,0.1
1,Desktop,7144,6.2,0.2
2,Tablet,4777,4.1,0.6
3,Mobile,20853,1.3,2.1
4,Unknown,29,0.0,3.4


In [24]:
run(22)

Q22 - Studio engagement depth
  Business: which studios create fans (repeat viewing), not just impressions.
  Technique: CTE computing two grains (events, viewers) combined into a ratio.

WITH studio_stats AS (
    SELECT c.Studio,
           COUNT(*)                  AS views,
           COUNT(DISTINCT f.User_ID) AS viewers,
           AVG(f.Completion_Percentage) AS avg_completion
    FROM fact_watch_events f
    JOIN dim_content c USING (Content_ID)
    GROUP BY c.Studio
)
SELECT Studio, views, viewers,
       ROUND(1.0 * views / viewers, 2)  AS views_per_viewer,
       ROUND(avg_completion, 1)         AS avg_completion_pct
FROM studio_stats
WHERE viewers >= 300
ORDER BY views_per_viewer DESC
LIMIT 10;



,Studio,views,viewers,views_per_viewer,avg_completion_pct
0,MAPPA,6921,3171,2.18,62.2
1,Production I.G,1252,639,1.96,61.8
2,Toei Animation,5100,2631,1.94,62.4
3,A-1 Pictures,3167,1732,1.83,61.3
4,Madhouse,4526,2498,1.81,61.9
5,Sunrise,825,482,1.71,61.2
6,CloverWorks,1467,880,1.67,60.3
7,8bit,1110,683,1.63,61.0
8,Wit Studio,1271,787,1.61,61.1
9,Bones,2834,1795,1.58,61.1


**Part 2 takeaways.** Shonen fans are the largest revenue pool but not the
richest per head (Q12) — niche-genre fans often pay more per user. Completion
leaderboards (Q15) reward holding attention rather than attracting clicks, and
the silent-active cohort (Q19) is a churn early-warning list you can hand to
CRM today. Q21 shows Smart TV as the finishing device from pure SQL.

---
## Part 3 · Window functions

The differentiator tier in interviews: ranking within groups, reaching
neighbouring rows (LAG/LEAD), running totals and rolling averages — all
without self-joins.

In [25]:
run(23)

Q23 - Each user's most recent watch event
  Business: "last seen watching" — feeds win-back campaigns and resume rows.
  Technique: ROW_NUMBER() PARTITION BY user, keep rn = 1. The canonical
  "latest row per entity" pattern.

WITH latest AS (
    SELECT f.User_ID, f.Watch_Date, c.Anime_Title, f.Episode_Number,
           ROW_NUMBER() OVER (PARTITION BY f.User_ID
                              ORDER BY f.Watch_Date DESC) AS rn
    FROM fact_watch_events f
    JOIN dim_content c USING (Content_ID)
)
SELECT User_ID, Watch_Date, Anime_Title, Episode_Number
FROM latest
WHERE rn = 1
ORDER BY Watch_Date DESC
LIMIT 10;



,User_ID,Watch_Date,Anime_Title,Episode_Number
0,U07407,2026-06-29 23:59:00,Hell's Paradise,7
1,U03130,2026-06-29 23:56:00,One Piece,557
2,U06597,2026-06-29 23:56:00,Ouran High School Host Club,3
3,U07738,2026-06-29 23:51:00,One Piece,791
4,U06610,2026-06-29 23:49:00,Kimi ni Todoke,19
5,U00323,2026-06-29 23:46:00,Dr. Stone,39
6,U00153,2026-06-29 23:43:00,My Hero Academia,11
7,U00234,2026-06-29 23:43:00,My Dress-Up Darling,8
8,U03045,2026-06-29 23:26:00,One Piece,750
9,U03780,2026-06-29 22:49:00,Jujutsu Kaisen,20


In [26]:
run(24)

Q24 - Top 3 titles inside every genre
  Business: genre-level renewal shortlists in one pass, not 12 queries.
  Technique: RANK() partitioned by genre over an aggregated CTE.

WITH genre_hours AS (
    SELECT c.Genre, c.Anime_Title,
           SUM(f.Watch_Time_Minutes) / 60.0 AS hours
    FROM fact_watch_events f
    JOIN dim_content c USING (Content_ID)
    GROUP BY c.Genre, c.Anime_Title
),
ranked AS (
    SELECT Genre, Anime_Title, hours,
           RANK() OVER (PARTITION BY Genre ORDER BY hours DESC) AS genre_rank
    FROM genre_hours
)
SELECT Genre, genre_rank, Anime_Title, ROUND(hours, 0) AS watch_hours
FROM ranked
WHERE genre_rank <= 3
ORDER BY Genre, genre_rank;



,Genre,genre_rank,Anime_Title,watch_hours
0,Comedy,1,Gintama,68.0
1,Comedy,2,Mob Psycho 100,60.0
2,Comedy,3,The Disastrous Life of Saiki K.,58.0
3,Fantasy,1,Solo Leveling,281.0
4,Fantasy,2,Frieren: Beyond Journey's End,233.0
5,Fantasy,3,Made in Abyss,72.0
6,Horror,1,Parasyte: The Maxim,60.0
7,Horror,2,Another,57.0
8,Horror,3,Higurashi: When They Cry,56.0
9,Isekai,1,Sword Art Online,153.0


In [27]:
run(25)

Q25 - Studios ranked by viewer rating
  Business: a quality league table for licensing negotiations.
  Technique: DENSE_RANK (no gaps on ties) over a HAVING-guarded aggregate.

WITH studio_ratings AS (
    SELECT c.Studio,
           COUNT(f.User_Rating)         AS ratings,
           AVG(f.User_Rating)           AS avg_rating
    FROM fact_watch_events f
    JOIN dim_content c USING (Content_ID)
    WHERE f.User_Rating IS NOT NULL
    GROUP BY c.Studio
    HAVING COUNT(f.User_Rating) >= 200
)
SELECT DENSE_RANK() OVER (ORDER BY avg_rating DESC) AS quality_rank,
       Studio, ratings, ROUND(avg_rating, 2) AS avg_rating
FROM studio_ratings
ORDER BY quality_rank;



,quality_rank,Studio,ratings,avg_rating
0,1,Toei Animation,2346,8.11
1,2,MAPPA,3196,8.03
2,3,Madhouse,2075,7.99
3,4,Ufotable,1016,7.94
4,5,Doga Kobo,240,7.93
5,6,Studio Trigger,244,7.88
6,7,White Fox,429,7.79
7,8,Wit Studio,564,7.77
8,9,Sunrise,375,7.76
9,10,Kinema Citrus,311,7.60


In [28]:
run(26).tail(12)

Q26 - Month-over-month sign-up growth
  Business: is acquisition accelerating or stalling?
  Technique: LAG() to reach the previous row; growth % from current vs LAG.

WITH monthly AS (
    SELECT strftime('%Y-%m', Subscription_Start_Date) AS month,
           COUNT(*) AS signups
    FROM fact_subscriptions
    GROUP BY month
)
SELECT month, signups,
       LAG(signups) OVER (ORDER BY month)             AS prev_month,
       ROUND(100.0 * (signups - LAG(signups) OVER (ORDER BY month))
             / LAG(signups) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM monthly
ORDER BY month;



,month,signups,prev_month,mom_growth_pct
30,2025-07,279,244.0,14.3
31,2025-08,272,279.0,-2.5
32,2025-09,270,272.0,-0.7
33,2025-10,297,270.0,10.0
34,2025-11,319,297.0,7.4
35,2025-12,330,319.0,3.4
36,2026-01,330,330.0,0.0
37,2026-02,281,330.0,-14.8
38,2026-03,308,281.0,9.6
39,2026-04,312,308.0,1.3


In [29]:
run(27).tail(12)

Q27 - Net subscriber adds per month
  Business: the headline growth number: sign-ups minus cancellations.
  Technique: two CTEs LEFT JOINed on month (every month has sign-ups here,
  so signups is the safe spine).

WITH s AS (
    SELECT strftime('%Y-%m', Subscription_Start_Date) AS month, COUNT(*) AS signups
    FROM fact_subscriptions GROUP BY month
),
c AS (
    SELECT strftime('%Y-%m', Subscription_End_Date) AS month, COUNT(*) AS cancels
    FROM fact_subscriptions
    WHERE Subscription_End_Date IS NOT NULL GROUP BY month
)
SELECT s.month, s.signups,
       COALESCE(c.cancels, 0)              AS cancels,
       s.signups - COALESCE(c.cancels, 0)  AS net_adds
FROM s
LEFT JOIN c ON c.month = s.month
ORDER BY s.month;



,month,signups,cancels,net_adds
30,2025-07,279,140,139
31,2025-08,272,165,107
32,2025-09,270,157,113
33,2025-10,297,190,107
34,2025-11,319,175,144
35,2025-12,330,194,136
36,2026-01,330,218,112
37,2026-02,281,195,86
38,2026-03,308,198,110
39,2026-04,312,213,99


In [30]:
run(28).tail(12)

Q28 - Cumulative sign-ups (running total)
  Business: the "users ever acquired" curve for the growth deck.
  Technique: SUM() OVER (ORDER BY ...) — the running-total idiom.

WITH monthly AS (
    SELECT strftime('%Y-%m', Subscription_Start_Date) AS month, COUNT(*) AS signups
    FROM fact_subscriptions GROUP BY month
)
SELECT month, signups,
       SUM(signups) OVER (ORDER BY month) AS cumulative_signups
FROM monthly
ORDER BY month;



,month,signups,cumulative_signups
30,2025-07,279,4613
31,2025-08,272,4885
32,2025-09,270,5155
33,2025-10,297,5452
34,2025-11,319,5771
35,2025-12,330,6101
36,2026-01,330,6431
37,2026-02,281,6712
38,2026-03,308,7020
39,2026-04,312,7332


In [31]:
run(29).tail(12)

Q29 - Monthly watch hours with a 3-month rolling average
  Business: smooths seasonal spikes so the underlying trend is visible.
  Technique: AVG() OVER with ROWS BETWEEN 2 PRECEDING AND CURRENT ROW.

WITH monthly AS (
    SELECT strftime('%Y-%m', Watch_Date) AS month,
           SUM(Watch_Time_Minutes) / 60.0 AS watch_hours
    FROM fact_watch_events
    GROUP BY month
)
SELECT month,
       ROUND(watch_hours, 0) AS watch_hours,
       ROUND(AVG(watch_hours) OVER (ORDER BY month
                                    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 0) AS rolling_3m
FROM monthly
ORDER BY month;



,month,watch_hours,rolling_3m
30,2025-07,499.0,351.0
31,2025-08,339.0,374.0
32,2025-09,304.0,381.0
33,2025-10,564.0,403.0
34,2025-11,386.0,418.0
35,2025-12,356.0,435.0
36,2026-01,706.0,483.0
37,2026-02,382.0,481.0
38,2026-03,415.0,501.0
39,2026-04,720.0,506.0


In [32]:
run(30)

Q30 - Days between a user's consecutive watch sessions
  Business: viewing cadence per plan — habit strength predicts retention.
  Technique: LEAD() to reach the *next* event; julianday() for date math.

WITH gaps AS (
    SELECT User_ID, Watch_Date,
           LEAD(Watch_Date) OVER (PARTITION BY User_ID ORDER BY Watch_Date) AS next_watch
    FROM fact_watch_events
)
SELECT u.Subscription_Plan,
       ROUND(AVG(julianday(g.next_watch) - julianday(g.Watch_Date)), 1) AS avg_gap_days,
       COUNT(*) AS measured_gaps
FROM gaps g
JOIN dim_user u USING (User_ID)
WHERE g.next_watch IS NOT NULL
GROUP BY u.Subscription_Plan
ORDER BY avg_gap_days;



,Subscription_Plan,avg_gap_days,measured_gaps
0,Family,22.8,8278
1,Premium,27.0,14631
2,Basic,39.8,6328
3,Free,67.0,3816


In [33]:
run(31)

Q31 - Churn rate by engagement quartile
  Business: turns "engagement matters" into a number per quartile.
  Technique: NTILE(4) buckets users into equal quarters by score.

WITH quartiles AS (
    SELECT User_ID, Engagement_Score, Subscription_Status,
           NTILE(4) OVER (ORDER BY Engagement_Score) AS quartile
    FROM dim_user
)
SELECT quartile,
       COUNT(*)                                  AS users,
       ROUND(MIN(Engagement_Score), 1)           AS min_score,
       ROUND(MAX(Engagement_Score), 1)           AS max_score,
       ROUND(100.0 * SUM(CASE WHEN Subscription_Status = 'Cancelled'
                              THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_pct
FROM quartiles
GROUP BY quartile
ORDER BY quartile;



,quartile,users,min_score,max_score,churn_pct
0,1,1999,6.8,35.9,60.9
1,2,1998,35.9,49.0,59.1
2,3,1998,49.0,63.1,58.4
3,4,1998,63.1,95.5,45.2


In [34]:
run(32)

Q32 - Genre share of total watch time
  Business: portfolio concentration — how dependent are we on one genre?
  Technique: window over an aggregate — SUM(SUM(x)) OVER () gives the grand
  total on each grouped row without a second query.

SELECT c.Genre,
       ROUND(SUM(f.Watch_Time_Minutes) / 60.0, 0)          AS watch_hours,
       ROUND(100.0 * SUM(f.Watch_Time_Minutes)
             / SUM(SUM(f.Watch_Time_Minutes)) OVER (), 1)  AS share_pct
FROM fact_watch_events f
JOIN dim_content c USING (Content_ID)
GROUP BY c.Genre
ORDER BY watch_hours DESC;



,Genre,watch_hours,share_pct
0,Shonen,4874.0,50.3
1,Isekai,852.0,8.8
2,Fantasy,727.0,7.5
3,Seinen,705.0,7.3
4,Slice of Life,542.0,5.6
5,Romance,382.0,3.9
6,Thriller,344.0,3.6
7,Sports,324.0,3.4
8,Mecha,260.0,2.7
9,Shojo,254.0,2.6


In [35]:
run(33)

Q33 - The top-revenue customer in every country
  Business: local whales — candidates for VIP retention treatment.
  Technique: ROW_NUMBER() PARTITION BY country ORDER BY revenue.

WITH ranked AS (
    SELECT u.Country, u.User_ID, s.Subscription_Plan, s.Revenue,
           ROW_NUMBER() OVER (PARTITION BY u.Country
                              ORDER BY s.Revenue DESC) AS rn
    FROM fact_subscriptions s
    JOIN dim_user u USING (User_ID)
)
SELECT Country, User_ID, Subscription_Plan,
       ROUND(Revenue, 0) AS lifetime_revenue
FROM ranked
WHERE rn = 1
ORDER BY Revenue DESC;



,Country,User_ID,Subscription_Plan,lifetime_revenue
0,Japan,U07322,Family,600.0
1,Mexico,U01631,Family,585.0
2,South Korea,U02708,Family,585.0
3,UK,U04154,Family,585.0
4,Canada,U02171,Family,570.0
5,France,U02549,Family,570.0
6,Indonesia,U00621,Family,525.0
7,Philippines,U07104,Family,525.0
8,USA,U06485,Family,525.0
9,Brazil,U02980,Family,510.0


In [36]:
run(34)

Q34 - Gateway anime: the first title new users watch
  Business: acquisition content — what to put on the signed-out homepage.
  Technique: FIRST_VALUE() over each user's timeline, deduplicated.

WITH firsts AS (
    SELECT DISTINCT User_ID,
           FIRST_VALUE(Content_ID) OVER (PARTITION BY User_ID
                                         ORDER BY Watch_Date) AS first_content
    FROM fact_watch_events
)
SELECT c.Anime_Title, c.Genre,
       COUNT(*) AS users_started_here
FROM firsts
JOIN dim_content c ON c.Content_ID = firsts.first_content
GROUP BY c.Anime_Title, c.Genre
ORDER BY users_started_here DESC
LIMIT 10;



,Anime_Title,Genre,users_started_here
0,One Piece,Shonen,1017
1,Attack on Titan,Shonen,571
2,Demon Slayer: Kimetsu no Yaiba,Shonen,413
3,Jujutsu Kaisen,Shonen,366
4,My Hero Academia,Shonen,311
5,Naruto Shippuden,Shonen,286
6,Solo Leveling,Fantasy,229
7,Chainsaw Man,Shonen,211
8,Spy x Family,Slice of Life,182
9,Frieren: Beyond Journey's End,Fantasy,181


In [37]:
run(35)

Q35 - Daily active users with a 7-day rolling average (June 2026)
  Business: the ops heartbeat metric, smoothed for weekday/weekend noise.
  Technique: compute the window over a warm-up buffer (from May 25), then
  filter to June — filtering first would corrupt the first week's
  rolling values.

WITH daily AS (
    SELECT DATE(Watch_Date) AS day, COUNT(DISTINCT User_ID) AS dau
    FROM fact_watch_events
    WHERE Watch_Date >= '2026-05-25'
    GROUP BY day
),
windowed AS (
    SELECT day, dau,
           AVG(dau) OVER (ORDER BY day
                          ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS dau_7d
    FROM daily
)
SELECT day, dau, ROUND(dau_7d, 1) AS dau_7d_avg
FROM windowed
WHERE day >= '2026-06-01'
ORDER BY day;



,day,dau,dau_7d_avg
0,2026-06-01,48,48.4
1,2026-06-02,41,49.9
2,2026-06-03,38,50.7
3,2026-06-04,43,52.9
4,2026-06-05,42,53.1
5,2026-06-06,92,53.7
6,2026-06-07,80,54.9
7,2026-06-08,39,53.6
8,2026-06-09,30,52.0
9,2026-06-10,42,52.6


In [38]:
run(36)

Q36 - Monthly active users and their growth (last 12 months)
  Business: MAU is the investor metric; MoM growth is its derivative.
  Technique: COUNT(DISTINCT) per month + LAG on the aggregated CTE.

WITH mau AS (
    SELECT strftime('%Y-%m', Watch_Date) AS month,
           COUNT(DISTINCT User_ID) AS mau
    FROM fact_watch_events
    GROUP BY month
)
SELECT month, mau,
       ROUND(100.0 * (mau - LAG(mau) OVER (ORDER BY month))
             / LAG(mau) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM mau
ORDER BY month DESC
LIMIT 12;



,month,mau,mom_growth_pct
0,2026-06,1527,28.1
1,2026-05,1192,-27.2
2,2026-04,1637,61.3
3,2026-03,1015,8.7
4,2026-02,934,-38.3
5,2026-01,1514,69.0
6,2025-12,896,-3.4
7,2025-11,928,-23.1
8,2025-10,1207,60.1
9,2025-09,754,-9.6


**Part 3 takeaways.** MoM growth (Q26) is noisy while the running total (Q28)
and 3-month rolling average (Q29) tell the clean story — that pairing is the
classic "smooth the derivative" interview answer. Q30 shows paid users return
days sooner than free users; Q34's "gateway anime" list is an acquisition
insight most teams never think to query. Q35 demonstrates a subtle trap:
compute rolling windows *before* filtering the date range, or the first week
lies.

---
## Part 4 · Business KPI queries

Executive metrics expressed directly in SQL — the layer a BI tool would sit on
top of.

In [39]:
run(37)

Q37 - Executive KPI snapshot (one row)
  Business: the numbers a CEO asks for before coffee.
  Technique: scalar subqueries composed into a single-row dashboard feed.

SELECT (SELECT COUNT(*) FROM dim_user)                                       AS total_users,
       (SELECT COUNT(*) FROM dim_user WHERE Subscription_Status = 'Active')  AS active_users,
       (SELECT ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dim_user), 1)
        FROM dim_user WHERE Subscription_Status = 'Cancelled')               AS churn_rate_pct,
       (SELECT ROUND(SUM(Revenue), 0) FROM fact_subscriptions)               AS lifetime_revenue,
       (SELECT ROUND(SUM(Monthly_Fee), 0) FROM fact_subscriptions
        WHERE Subscription_Status = 'Active')                                AS current_mrr,
       (SELECT ROUND(SUM(Watch_Time_Minutes) / 60.0, 0)
        FROM fact_watch_events)                                              AS total_watch_hours,
       (SELECT ROUND(AVG(Customer_Satisfaction), 2) FROM dim_

,total_users,active_users,churn_rate_pct,lifetime_revenue,current_mrr,total_watch_hours,avg_csat
0,7993,3524,55.9,374694.0,22551.0,9681.0,7.07


In [40]:
run(38)

Q38 - Monthly churn hazard by tenure bucket
  Business: proves the risk window is months 1-3 — where onboarding money goes.
  Technique: a VALUES CTE of bucket bounds drives correlated subqueries;
  each bucket's cancellations are normalised by the users who
  survived long enough to be at risk, per month in the bucket.

WITH bounds (bucket, lo, hi) AS (
    VALUES ('01-03m', 1, 3), ('04-06m', 4, 6), ('07-12m', 7, 12),
           ('13-24m', 13, 24), ('25m+', 25, 42)
)
SELECT b.bucket,
       (SELECT COUNT(*) FROM fact_subscriptions
        WHERE Subscription_Status = 'Cancelled'
          AND Membership_Tenure BETWEEN b.lo AND b.hi)     AS cancellations,
       (SELECT COUNT(*) FROM fact_subscriptions
        WHERE Membership_Tenure >= b.lo)                   AS users_at_risk,
       ROUND(100.0 * (SELECT COUNT(*) FROM fact_subscriptions
                      WHERE Subscription_Status = 'Cancelled'
                        AND Membership_Tenure BETWEEN b.lo AND b.hi)
             / (SEL

,bucket,cancellations,users_at_risk,monthly_hazard_pct
0,01-03m,2619,7993,10.92
1,04-06m,778,4391,5.91
2,07-12m,623,3065,3.39
3,13-24m,384,1667,1.92
4,25m+,65,422,0.86


In [41]:
run(39)

Q39 - ARPU and revenue mix by plan
  Business: which tier actually funds the platform.
  Technique: window share-of-total on top of a grouped aggregate.

SELECT s.Subscription_Plan,
       COUNT(*)                                            AS users,
       ROUND(AVG(s.Revenue_Per_Month), 2)                  AS arpu_monthly,
       ROUND(SUM(s.Revenue), 0)                            AS lifetime_revenue,
       ROUND(100.0 * SUM(s.Revenue)
             / SUM(SUM(s.Revenue)) OVER (), 1)             AS revenue_share_pct
FROM fact_subscriptions s
GROUP BY s.Subscription_Plan
ORDER BY lifetime_revenue DESC;



,Subscription_Plan,users,arpu_monthly,lifetime_revenue,revenue_share_pct
0,Premium,1992,9.99,175235.0,46.8
1,Family,788,14.99,115303.0,30.8
2,Basic,2018,4.99,69865.0,18.6
3,Free,3195,0.68,14291.0,3.8


In [42]:
run(40)

Q40 - Free vs paid behaviour gap
  Business: the conversion pitch — what paying unlocks, in user behaviour.
  Technique: CASE collapses four plans into two cohorts inside one pass.

SELECT CASE WHEN u.Subscription_Plan = 'Free' THEN 'Free' ELSE 'Paid' END AS cohort,
       COUNT(DISTINCT u.User_ID)                       AS users,
       ROUND(1.0 * COUNT(*) / COUNT(DISTINCT u.User_ID), 1) AS events_per_user,
       ROUND(AVG(f.Completion_Percentage), 1)          AS avg_completion_pct,
       ROUND(AVG(f.Watch_Time_Minutes), 1)             AS avg_watch_min,
       ROUND(100.0 * AVG(f.Ad_Shown), 1)               AS ad_exposure_pct
FROM fact_watch_events f
JOIN dim_user u USING (User_ID)
GROUP BY cohort;



,cohort,users,events_per_user,avg_completion_pct,avg_watch_min,ad_exposure_pct
0,Free,3195,2.2,54.0,12.4,91.7
1,Paid,4798,7.1,63.1,14.5,0.0


In [43]:
run(41)

Q41 - Prime-time share by device
  Business: where evening ad inventory and simulcast pushes should go.
  Technique: AVG over a 0/1 flag is a percentage — the cleanest share idiom.

SELECT Device,
       COUNT(*)                              AS events,
       ROUND(100.0 * AVG(Peak_Hour), 1)      AS prime_time_share_pct,
       ROUND(100.0 * AVG(Weekend_Viewing), 1) AS weekend_share_pct
FROM fact_watch_events
GROUP BY Device
ORDER BY events DESC;



,Device,events,prime_time_share_pct,weekend_share_pct
0,Mobile,20853,48.6,49.3
1,Smart TV,8243,48.5,48.2
2,Desktop,7144,48.4,49.2
3,Tablet,4777,48.0,49.6
4,Unknown,29,75.9,62.1


In [44]:
run(42)

Q42 - Weekend vs weekday intensity (per calendar day)
  Business: raw weekend totals mislead — there are only 2 weekend days per 7.
  Per-day normalisation shows true intensity.
  Technique: JOIN to dim_date and divide by COUNT(DISTINCT Date_ID).

SELECT CASE d.Is_Weekend WHEN 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
       COUNT(*)                                       AS events,
       COUNT(DISTINCT d.Date_ID)                      AS calendar_days,
       ROUND(1.0 * COUNT(*) / COUNT(DISTINCT d.Date_ID), 0) AS events_per_day,
       ROUND(AVG(f.Watch_Time_Minutes), 1)            AS avg_watch_min
FROM fact_watch_events f
JOIN dim_date d USING (Date_ID)
GROUP BY day_type;



,day_type,events,calendar_days,events_per_day,avg_watch_min
0,Weekday,20881,830,25.0,14.2
1,Weekend,20165,349,58.0,14.1


In [45]:
run(43)

Q43 - Sign-up cohorts: who is still with us
  Business: cohort survival — is retention improving for newer vintages?
  (Newer cohorts always look better; compare like-for-like ages.)
  Technique: string-built quarter labels from strftime parts.

SELECT strftime('%Y', Subscription_Start_Date) || '-Q' ||
       ((CAST(strftime('%m', Subscription_Start_Date) AS INTEGER) + 2) / 3) AS signup_cohort,
       COUNT(*)                                        AS users,
       ROUND(100.0 * SUM(CASE WHEN Subscription_Status = 'Active'
                              THEN 1 ELSE 0 END) / COUNT(*), 1) AS still_active_pct,
       ROUND(AVG(Membership_Tenure), 1)                AS avg_tenure_months
FROM fact_subscriptions
GROUP BY signup_cohort
ORDER BY signup_cohort;



,signup_cohort,users,still_active_pct,avg_tenure_months
0,2023-Q1,75,17.3,16.5
1,2023-Q2,173,17.3,13.5
2,2023-Q3,235,21.3,13.9
3,2023-Q4,328,23.2,12.8
4,2024-Q1,436,24.1,11.8
5,2024-Q2,474,27.6,12.3
6,2024-Q3,575,32.5,11.2
7,2024-Q4,576,31.3,9.6
8,2025-Q1,687,35.7,8.7
9,2025-Q2,775,39.6,7.6


In [46]:
run(44)

Q44 - Acquisition channel quality
  Business: marketing spend should follow paid conversion and retention,
  not raw sign-up counts.
  Technique: several conditional aggregates profiling each channel in one pass.

SELECT u.Referral_Source,
       COUNT(*)                                        AS users,
       ROUND(100.0 * SUM(CASE WHEN u.Subscription_Plan <> 'Free'
                              THEN 1 ELSE 0 END) / COUNT(*), 1) AS paid_share_pct,
       ROUND(100.0 * SUM(CASE WHEN u.Subscription_Status = 'Cancelled'
                              THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_pct,
       ROUND(AVG(u.Engagement_Score), 1)               AS avg_engagement
FROM dim_user u
GROUP BY u.Referral_Source
ORDER BY paid_share_pct DESC;



,Referral_Source,users,paid_share_pct,churn_pct,avg_engagement
0,Advertisement,939,61.4,54.8,49.8
1,YouTube,1131,61.1,58.4,49.5
2,Social Media,2205,60.9,56.1,50.2
3,Friend Referral,1741,59.6,54.8,49.9
4,Google Search,1477,58.2,55.9,50.1
5,App Store,500,58.0,55.4,50.6


In [47]:
run(45)

Q45 - Support load vs satisfaction and churn
  Business: every ticket bucket step costs satisfaction — quantify the link
  between support pain and cancellations.
  Technique: CASE bucketing + three conditional aggregates.

SELECT CASE WHEN Support_Tickets = 0 THEN '0 tickets'
            WHEN Support_Tickets <= 2 THEN '1-2 tickets'
            ELSE '3+ tickets' END                      AS ticket_bucket,
       COUNT(*)                                        AS users,
       ROUND(AVG(Customer_Satisfaction), 2)            AS avg_csat,
       ROUND(100.0 * SUM(CASE WHEN Subscription_Status = 'Cancelled'
                              THEN 1 ELSE 0 END) / COUNT(*), 1) AS churn_pct
FROM dim_user
GROUP BY ticket_bucket
ORDER BY ticket_bucket;



,ticket_bucket,users,avg_csat,churn_pct
0,0 tickets,3929,7.47,54.5
1,1-2 tickets,3651,6.78,57.2
2,3+ tickets,413,5.75,57.4


**Part 4 takeaways.** The snapshot (Q37) gives the one-row heartbeat; the
hazard table (Q38) proves the retention war is won or lost in months 1–3;
ARPU by plan (Q39) shows Premium as the revenue engine with Family close
behind per account. Channel quality (Q44) reorders marketing priorities the
moment you sort by paid share instead of volume, and Q45 puts a churn number
on every support-ticket bucket.

---
## Coverage checklist

GROUP BY ✓ · HAVING ✓ · CASE ✓ · subqueries ✓ · `NOT EXISTS` anti-join ✓ ·
CTEs ✓ · ROW_NUMBER ✓ · RANK ✓ · DENSE_RANK ✓ · NTILE ✓ · LAG ✓ · LEAD ✓ ·
FIRST_VALUE ✓ · running totals ✓ · rolling averages ✓ · monthly trends ✓ ·
top-N per group ✓ · cohorts ✓ · DAU/MAU ✓

**Next:** module 5 formalises the KPI layer (`docs/kpi_definitions.md`) and
exports the Power BI feed.

In [48]:
conn.close()